# **Music Transformer STEP3 - Music Generation**


In [1]:
!pip install -q torch torchaudio encodec torchcodec


[notice] A new release of pip is available: 23.3.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import torch
import torchaudio
import encodec

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

PyTorch version: 2.9.1+cu128
CUDA available: True


In [3]:
import os
os.environ["TORCH_HOME"] = "/mnt/workspace/torch_cache"

### **生成**

In [4]:
import torch
import torch.nn as nn
from music_transformer import MusicTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"

model = MusicTransformer(
    vocab_size=1024,
    num_codebooks=4,
    embed_dim=512,
    max_seq_len=8192,
    num_layers=6,
    num_heads=8,
    dropout=0.2,
).to(device)

checkpoint = torch.load("maestro_checkpoints/checkpoint_epoch4.pt", map_location=device)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()


MusicTransformer(
  (embeddings): ModuleList(
    (0-3): 4 x Embedding(1024, 512)
  )
  (pos_embedding): Embedding(8192, 512)
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-5): 6 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=512, out_features=512, bias=True)
        )
        (linear1): Linear(in_features=512, out_features=2048, bias=True)
        (dropout): Dropout(p=0.2, inplace=False)
        (linear2): Linear(in_features=2048, out_features=512, bias=True)
        (norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.2, inplace=False)
        (dropout2): Dropout(p=0.2, inplace=False)
      )
    )
  )
  (output_head): Linear(in_features=512, out_features=1024, bias=True)
)

In [7]:
import torch.nn.functional as F

@torch.no_grad()
def generate(model, start_tokens, max_new_tokens=200, temperature=1.0):
    model.eval()
    x = start_tokens.to(device)

    for _ in range(max_new_tokens):

        x_cond = x[:, -model.max_seq_len:]

        logits = model(x_cond)  # [B, T, K, vocab]

        next_tokens = []

        for k in range(model.num_codebooks):
            last_logits = logits[:, -1, k, :] / temperature
            probs = F.softmax(last_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            next_tokens.append(next_token)

        next_tokens = torch.stack(next_tokens, dim=-1)  # [B, 1, K]

        x = torch.cat([x, next_tokens], dim=1)

    return x

In [20]:
from dataset import MusicTokenDataset

dataset = MusicTokenDataset(
    "maestro_tokens_3kbps",
    block_size=1024,
    stride=256
)

x_sample, _ = dataset[0]
start = x_sample[:10].unsqueeze(0)  # 前10帧作为prompt

generated_tokens = generate(
    model,
    start_tokens=start,
    max_new_tokens=400,
    temperature=1.1
)

for k in range(4):
    print("Codebook", k, 
          generated_tokens[0, :, k].unique().shape[0])


Codebook 0 144
Codebook 1 230
Codebook 2 260
Codebook 3 282


这里已经完成了生成。我们可以看到每个 Codebook 被用了多少不同的音频 token。

调节 temperature 并多尝试几次，可以让每个 Codebook 使用的 token 更加多样，从而提高生成结果的丰富性。

### **解码**

In [21]:
from encodec import EncodecModel
import torchaudio

codec = EncodecModel.encodec_model_24khz()
codec.set_target_bandwidth(3.0)  # 3kbps
codec = codec.to(device)
codec.eval()

codes = generated_tokens[0].permute(1, 0).unsqueeze(0).to(device)

encoded_frames = [(codes, None)]  # 关键！！！

with torch.no_grad():
    wav = codec.decode(encoded_frames)

/usr/local/lib/python3.11/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


In [22]:
from IPython.display import Audio

audio = wav[0, 0].cpu().numpy()

Audio(audio, rate=24000)

试听生成的音乐。生成到自己满意的音乐时运行下面 Cell 保存 .wav 文件。

#### **保存**

In [24]:
import soundfile as sf

audio = wav[0, 0].cpu().numpy()

sf.write("generated_music.wav", audio, 24000)